# Pre-process Notes
This notebook examines and pre-processes the DAX and Doctor notes data.

In [ ]:
import pandas as pd
import re

In [ ]:
from project_config import BIAS_NOTES_RAW_CSV

file_path = BIAS_NOTES_RAW_CSV
if not file_path:
    raise RuntimeError("Set BIAS_NOTES_RAW_CSV in .env before running this notebook.")

# Load the CSV file
df = pd.read_csv(file_path)
print(f"DataFrame shape: {df.shape}")


In [ ]:
# Display column names and types
print("Columns:")
print(df.columns.tolist())
print("\nData types:")
print(df.dtypes)

In [ ]:
# Display first few rows
df.head()

In [ ]:
# Basic statistics
print("Basic Info:")
print(f"Total rows: {len(df)}")
print(f"\nMissing values per column:")
print(df.isnull().sum())

## DAX Text Analysis

In [ ]:
# Count cells containing the word "DAX" (case-insensitive)
dax_mask = df['note_text'].astype(str).str.contains('DAX', case=False, na=False)
dax_count = dax_mask.sum()

print(f"Number of note_text cells containing 'DAX': {dax_count}")
print(f"Percentage of total: {dax_count / len(df) * 100:.1f}%")

In [ ]:
# Count cells containing "Attestation      DAX copilot was used during"
attestation_pattern = r'Attestation\s+DAX copilot was used during'
attestation_mask = df['note_text'].astype(str).str.contains(attestation_pattern, case=False, na=False, regex=True)
attestation_count = attestation_mask.sum()

print(f"Number of note_text cells containing 'Attestation ... DAX copilot was used during': {attestation_count}")
print(f"Percentage of total: {attestation_count / len(df) * 100:.1f}%")

In [ ]:
# Analyze the maximum number of characters following "DAX copilot was used during"
pattern = r'DAX copilot was used during(.*)$'

def get_trailing_text_length(text):
    """Extract text after 'DAX copilot was used during' and return its length."""
    text = str(text)
    match = re.search(pattern, text, re.IGNORECASE | re.DOTALL)
    if match:
        return len(match.group(1))
    return None

# Get trailing text lengths for all matching cells
trailing_lengths = df['note_text'].apply(get_trailing_text_length).dropna()

if len(trailing_lengths) > 0:
    print(f"Number of cells with 'DAX copilot was used during': {len(trailing_lengths)}")
    print(f"\nCharacters following 'DAX copilot was used during':")
    print(f"  Maximum: {int(trailing_lengths.max())}")
    print(f"  Minimum: {int(trailing_lengths.min())}")
    print(f"  Mean: {trailing_lengths.mean():.1f}")
    print(f"  Median: {trailing_lengths.median():.1f}")
    print(f"\nDistribution of trailing character counts:")
    print(trailing_lengths.value_counts().sort_index())
else:
    print("No cells found containing 'DAX copilot was used during'")

In [ ]:
# Preview what the trailing text looks like
def get_trailing_text(text):
    """Extract text after 'DAX copilot was used during'."""
    text = str(text)
    match = re.search(pattern, text, re.IGNORECASE | re.DOTALL)
    if match:
        return match.group(1)
    return None

# Show unique trailing texts
trailing_texts = df['note_text'].apply(get_trailing_text).dropna().unique()
print(f"Unique trailing text patterns ({len(trailing_texts)} total):")
for i, text in enumerate(trailing_texts[:10]):  # Show first 10
    print(f"  {i+1}. '{text[:100]}{'...' if len(text) > 100 else ''}'")

In [ ]:
# Frequency distribution of DAX attestation phrases (normalized)
# This identifies all patterns that reveal DAX assisted with note generation

from collections import Counter

def extract_dax_phrases(text):
    """Extract and normalize DAX-related phrases from the text."""
    text = str(text)
    # Normalize: replace newlines and multiple spaces with single space
    text_normalized = re.sub(r'\s+', ' ', text)
    
    # Patterns to capture DAX attestation phrases
    # These patterns capture the full attestation sentence/phrase
    patterns = [
        r'(Note\s+written\s+with\s+DAX\s+Copilot[^.]*\.?)',
        r'(Attestation\s+DAX\s+copilot\s+was\s+used[^.]*\.?)',
        r'(DAX\s+copilot\s+was\s+used[^.]*\.?)',
        r'(This\s+note\s+has\s+been\s+written\s+with\s+DAX[^.]*\.?)',
        r'(Contains\s+text\s+generated\s+by\s+DAX[^.]*\.?)',
        r'(HPI\s+created\s+by\s+Nuance\s+DAX[^.]*:?)',
        r'(written\s+with\s+DAX\s+CoPilot[^.]*\.?)',
    ]
    
    found_phrases = []
    for pattern in patterns:
        matches = re.findall(pattern, text_normalized, re.IGNORECASE)
        for match in matches:
            # Normalize the match further
            normalized_match = re.sub(r'\s+', ' ', match).strip()
            found_phrases.append(normalized_match)
    
    return found_phrases

# Collect all DAX phrases
all_dax_phrases = []
for text in df['note_text']:
    all_dax_phrases.extend(extract_dax_phrases(text))

# Normalize further for counting (collapse minor variations)
def normalize_for_counting(phrase):
    """Collapse variations like different names into a single pattern."""
    # Replace <PERSON> placeholders and similar
    normalized = re.sub(r'<[A-Z_]+>', '<PLACEHOLDER>', phrase)
    # Normalize whitespace
    normalized = re.sub(r'\s+', ' ', normalized).strip()
    return normalized

normalized_phrases = [normalize_for_counting(p) for p in all_dax_phrases]
phrase_counts = Counter(normalized_phrases)

print(f"Total DAX attestation phrases found: {len(all_dax_phrases)}")
print(f"Unique patterns (after normalization): {len(phrase_counts)}")
print(f"\n{'='*80}")
print("FREQUENCY DISTRIBUTION OF DAX ATTESTATION PHRASES:")
print(f"{'='*80}\n")

for phrase, count in phrase_counts.most_common():
    print(f"[Count: {count:3d}] \"{phrase}\"")

print(f"\n{'='*80}")
print("SUMMARY OF PATTERNS TO REMOVE:")
print(f"{'='*80}")
print("""
Based on the above, the main patterns to remove are:
1. "Note written with DAX Copilot and patient..." (and variations)
2. "Attestation DAX copilot was used during this visit with patient consent."
3. "This note has been written with DAX CoPilot and edited as appropriate"
4. "Contains text generated by DAX Copilot"
5. "HPI created by Nuance DAX Copilot and..."
""")

## Clean Notes and Save

In [ ]:
# Remove DAX attestation sentences from note_text
# Keeps the rest of the content intact (e.g., "She has been gaining weight" is preserved)

def remove_dax_attestations(text):
    """Remove DAX attestation sentences while preserving other content."""
    if pd.isna(text):
        return text
    
    text = str(text)
    
    # Patterns to remove (order matters - more specific patterns first)
    # Each pattern removes the full sentence/phrase only
    removal_patterns = [
        # "Note written with DAX Copilot and patient <name>." (handles newlines in text)
        r'\s*Note\s+written\s+with\s+DAX\s+Copilot\s+and\s+patient\s+[^.]*\.?\s*',
        
        # "Attestation      DAX copilot was used during this visit with patient consent."
        r'\s*Attestation\s+DAX\s+copilot\s+was\s+used\s+during\s+this\s+visit\s+with\s+patient\s+consent\.?\s*',
        
        # "DAX copilot was used during this visit with patient consent." (without Attestation prefix)
        r'\s*DAX\s+copilot\s+was\s+used\s+during\s+this\s+visit\s+with\s+patient\s+consent\.?\s*',
        
        # "This note has been written with DAX CoPilot and edited as appropriate"
        r'\s*This\s+note\s+has\s+been\s+written\s+with\s+DAX\s+CoPilot\s+and\s+edited\s+as\s+appropriate\.?\s*',
        
        # "Contains text generated by DAX Copilot"
        r'\s*Contains\s+text\s+generated\s+by\s+DAX\s+Copilot\.?\s*',
        
        # "HPI created by Nuance DAX Copilot and <name>, <name>:" - remove only up to the colon
        r'\s*HPI\s+created\s+by\s+Nuance\s+DAX\s+Copilot\s+and\s+[^:]*:\s*',
    ]
    
    cleaned_text = text
    for pattern in removal_patterns:
        cleaned_text = re.sub(pattern, ' ', cleaned_text, flags=re.IGNORECASE)
    
    # Clean up extra whitespace
    cleaned_text = re.sub(r'\s+', ' ', cleaned_text).strip()
    
    return cleaned_text

# Apply cleaning to create new column
df['note_text_cleaned'] = df['note_text'].apply(remove_dax_attestations)

# Verify the cleaning worked
print("Cleaning complete!")
print(f"\nVerification - Notes still containing 'DAX' after cleaning:")
still_has_dax = df['note_text_cleaned'].astype(str).str.contains('DAX', case=False, na=False).sum()
print(f"  Count: {still_has_dax}")

if still_has_dax > 0:
    print("\n  Remaining DAX references (may need manual review):")
    remaining = df[df['note_text_cleaned'].astype(str).str.contains('DAX', case=False, na=False)]['note_text_cleaned']
    for i, text in enumerate(remaining.head(5)):
        # Find the DAX context
        match = re.search(r'.{0,50}DAX.{0,50}', text, re.IGNORECASE)
        if match:
            print(f"    {i+1}. \"...{match.group()}...\"")

In [ ]:
# Preview before/after comparison
print("BEFORE/AFTER COMPARISON (first 3 notes with DAX removed):\n")
print("="*80)

# Find notes that had DAX content removed
changed_mask = df['note_text'].astype(str) != df['note_text_cleaned'].astype(str)
changed_indices = df[changed_mask].index[:3]

for idx in changed_indices:
    original = str(df.loc[idx, 'note_text'])
    cleaned = str(df.loc[idx, 'note_text_cleaned'])
    
    print(f"\n--- Row {idx} ---")
    print(f"ORIGINAL (last 300 chars):\n  ...{original[-300:]}")
    print(f"\nCLEANED (last 300 chars):\n  ...{cleaned[-300:]}")
    print("-"*80)

## Repair Mojibake / Encoding Artifacts

Some source notes contain UTF-8 mojibake (e.g., `kg/m²` rendered as `kg/mÃ¢Â²`). `ftfy.fix_text` repairs these in place so the LLM sees correct characters.

In [ ]:
import ftfy
import re

# Known clinical-notes doubly-encoded artifacts that ftfy leaves with a stray prefix char.
_RESIDUAL_FIXES = [
    (re.compile(r'kg\s*/\s*m[\u00e2\u00c2]+\s*\u00b2'), 'kg/m\u00b2'),
    (re.compile(r'm[\u00e2\u00c2]+\s*\u00b2'), 'm\u00b2'),
]

def repair(text: str) -> str:
    fixed = ftfy.fix_text(text)
    for pat, repl in _RESIDUAL_FIXES:
        fixed = pat.sub(repl, fixed)
    return fixed

before_sample = df_cleaned.loc[df_cleaned['note_text'].astype(str).str.contains('\u00c3|\u00c2', regex=True, na=False), 'note_text'].head(1)
if not before_sample.empty:
    print('Mojibake sample BEFORE:')
    print('  ', before_sample.iloc[0][:200])

df_cleaned['note_text'] = df_cleaned['note_text'].astype(str).map(repair)

after_sample = df_cleaned.loc[df_cleaned['note_text'].astype(str).str.contains('kg/m', na=False), 'note_text'].head(1)
if not after_sample.empty:
    print('Sample AFTER:')
    print('  ', after_sample.iloc[0][:200])

remaining_mojibake = df_cleaned['note_text'].astype(str).str.contains('\u00c3|\u00c2', regex=True, na=False).sum()
print(f'Rows with remaining mojibake markers: {remaining_mojibake}')

In [ ]:
from project_config import BIAS_NOTES_CLEANED_CSV

# Save cleaned data to new CSV file
# Replace original note_text with cleaned version and save

df_cleaned = df.copy()
df_cleaned['note_text'] = df_cleaned['note_text_cleaned']
df_cleaned = df_cleaned.drop(columns=['note_text_cleaned'])

output_path = BIAS_NOTES_CLEANED_CSV
if not output_path:
    raise RuntimeError("Set BIAS_NOTES_CLEANED_CSV in .env before running this notebook.")

df_cleaned.to_csv(output_path, index=False)

print(f"Cleaned CSV saved to:
  {output_path}")
print(f"
File contains {len(df_cleaned)} rows")
print(f"Original columns preserved: {df_cleaned.columns.tolist()}")
